# Neysan detector on the frozen final model (one run, rules fixed in advance)

Runtime -> Change runtime type -> **T4 GPU**. Edit the three paths in the next cell, then **Runtime -> Run all**.
At the end, copy the block between `===== RESULT =====` and `===== END =====` and send it.

What it does (nothing here trains or changes the 8-class model):
1. loads the frozen checkpoint `swin_t_ft_aug_best.pt` exactly as the project does;
2. checks the pipeline: the 8-class validation result must be 640 / 659 correct;
3. takes the 768-number feature vector of every image (the input of the final layer);
4. trains a logistic regression "Neysan vs not Neysan" (train split + 70% of the confirmed Neysan images);
5. measures it with a FIXED threshold 0.5 (no tuning) on images it never saw.

Acceptance rules (fixed before running):
- holdout confirmed Neysan rejected >= 80%
- validation (659 ordinary images) rejected <= 1% (at most 6 images)

In [ ]:
# ---- edit these three paths (files on Google Drive) ----
DELIVERY_ZIP = "/content/drive/MyDrive/vehicle_classifier/vehicle-classifier_delivery_v1.zip"
DATASET_ZIP  = "/content/drive/MyDrive/vehicle_classifier/clean_dataset_verified.zip"
NEYSAN_ZIP   = "/content/drive/MyDrive/vehicle_classifier/neysan_images.zip"
OUT_JSON     = "/content/drive/MyDrive/vehicle_classifier/neysan_detector.json"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys, zipfile, hashlib, json, csv, random, time
EXPECTED = {DELIVERY_ZIP: "ccc9795e7397e96d2bad99899141320bf760bf2e6969d76c8fe8e734b565968b",
            DATASET_ZIP:  "a4cb2d8712d1a64ef6ef079e78ae135246f8135fa80c7cdfd4e8604443bb52ee",
            NEYSAN_ZIP:   "dca996f373faa130e08bc9b4700643e4ad04ce55a73ea428d02f3d15867e7956"}
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
for p, h in EXPECTED.items():
    assert os.path.isfile(p), f"missing: {p}"
    got = sha256(p)
    assert got == h, f"{os.path.basename(p)} is damaged or different (sha256 {got}); upload it again"
    print("OK", os.path.basename(p))
for p, d in [(DELIVERY_ZIP, "/content/pkg"), (DATASET_ZIP, "/content/data"), (NEYSAN_ZIP, "/content/neysan")]:
    if not os.path.isdir(d):
        with zipfile.ZipFile(p) as z: z.extractall(d)
REPO = "/content/pkg/vehicle-classifier"
sys.path.insert(0, REPO)
print("ready")

In [ ]:
import torch
from PIL import Image
from src.predict import load_model
from src.dataset import CLASSES
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, eval_transform, ckpt = load_model(os.path.join(REPO, "checkpoints", "swin_t_ft_aug_best.pt"))
model = model.to(device).eval()
feats = {}
model.net.head.register_forward_hook(lambda m, inp, out: feats.__setitem__("f", inp[0].detach()))
print("model:", ckpt["architecture"], "epoch", ckpt["epoch"], "| device:", device)

def embed(paths, batch=64):
    F, P = [], []
    for i in range(0, len(paths), batch):
        x = torch.stack([eval_transform(Image.open(p).convert("RGB")) for p in paths[i:i + batch]]).to(device)
        with torch.inference_mode():
            logits = model(x)
        F.append(feats["f"].float().cpu()); P.append(torch.softmax(logits.float(), 1).cpu())
    return torch.cat(F).numpy(), torch.cat(P).numpy()

def folder_images(split):
    out = []
    for c in CLASSES:
        for root, _, files in os.walk(f"/content/data/clean_dataset/{split}/{c}"):
            for f in sorted(files):
                out.append((os.path.join(root, f), c))
    return sorted(out)

train = folder_images("train"); val = folder_images("val")
ny = list(csv.DictReader(open("/content/neysan/neysan/neysan_manifest.csv", encoding="utf-8")))
assert (len(train), len(val), len(ny)) == (2633, 659, 621), (len(train), len(val), len(ny))
t0 = time.time()
Ftr, Ptr = embed([p for p, _ in train]); Fva, Pva = embed([p for p, _ in val])
Fny, Pny = embed([os.path.join("/content/neysan", r["zip_path"]) for r in ny])
print(f"features done in {time.time() - t0:.0f}s")

In [ ]:
import numpy as np
# pipeline check: the frozen model must reproduce its recorded validation result
yva = np.array([CLASSES.index(c) for _, c in val])
val_correct = int((Pva.argmax(1) == yva).sum())
print("8-class validation correct:", val_correct, "of 659 (expected 640)")
assert val_correct == 640, "pipeline mismatch - stop here and send this output"

# Neysan split, fixed in advance: confirmed images only, grouped by sha256, seed 42, 70 / 15 / 15
conf = [i for i, r in enumerate(ny) if r["basis"] == "CONFIRMED_NEYSAN"]
n1 = [i for i, r in enumerate(ny) if r["basis"] != "CONFIRMED_NEYSAN"]
shas = sorted({ny[i]["sha256"] for i in conf}); random.Random(42).shuffle(shas)
a, b = int(0.70 * len(shas)), int(0.85 * len(shas))
part = {s: ("train" if k < a else "calib" if k < b else "holdout") for k, s in enumerate(shas)}
ny_tr = [i for i in conf if part[ny[i]["sha256"]] == "train"]
ny_ca = [i for i in conf if part[ny[i]["sha256"]] == "calib"]
ny_ho = [i for i in conf if part[ny[i]["sha256"]] == "holdout"]
n1_clean = [i for i in n1 if ny[i]["sha256"] not in {ny[j]["sha256"] for j in ny_tr}]   # no copy of a training image
print("confirmed Neysan train / calib / holdout:", len(ny_tr), len(ny_ca), len(ny_ho), "| N1 check:", len(n1_clean))

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X = np.concatenate([Ftr, Fny[ny_tr]]); y = np.r_[np.zeros(len(Ftr)), np.ones(len(ny_tr))]
scaler = StandardScaler().fit(X)
clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000).fit(scaler.transform(X), y)
THRESHOLD = 0.5   # fixed, not tuned
def p_ny(F): return clf.predict_proba(scaler.transform(F))[:, 1]

pv = p_ny(Fva); rej_val = pv >= THRESHOLD
ho = (p_ny(Fny[ny_ho]) >= THRESHOLD).mean(); ca = (p_ny(Fny[ny_ca]) >= THRESHOLD).mean()
n1r = (p_ny(Fny[n1_clean]) >= THRESHOLD).mean()
vanet_idx = yva == CLASSES.index("vanet")
correct_kept = int(((Pva.argmax(1) == yva) & ~rej_val).sum())
ok = ho >= 0.80 and rej_val.sum() <= 6

In [ ]:
print("===== RESULT =====")
print(f"pipeline check: 8-class val correct {val_correct}/659")
print(f"holdout confirmed Neysan rejected: {ho:.3f}  ({int(round(ho*len(ny_ho)))}/{len(ny_ho)})   rule >= 0.80")
print(f"calib confirmed Neysan rejected:   {ca:.3f}  ({int(round(ca*len(ny_ca)))}/{len(ny_ca)})")
print(f"N1 (unreviewed) Neysan rejected:   {n1r:.3f}  (n={len(n1_clean)})")
print(f"validation images rejected:        {int(rej_val.sum())}/659   rule <= 6")
print(f"  of which vanet:                  {int(rej_val[vanet_idx].sum())}/{int(vanet_idx.sum())}")
print(f"  rejected per class:              { {c: int(rej_val[yva == k].sum()) for k, c in enumerate(CLASSES) if rej_val[yva == k].sum()} }")
print(f"8-class val correct after rejection: {correct_kept}/659 (was {val_correct})")
print("DECISION:", "PASS" if ok else "FAIL")
det = {"type": "logistic_regression_on_swin_head_input", "threshold": THRESHOLD, "feature_dim": int(Ftr.shape[1]),
       "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(), "coef": clf.coef_[0].tolist(),
       "intercept": float(clf.intercept_[0]), "checkpoint_sha256": "f03bacde0a155461ea4aede0b813b8688d6ed17973d5ad19b0d2c1b913ff64db",
       "trained_on": {"negatives": "train split 2633", "positives": f"confirmed Neysan {len(ny_tr)} (seed 42, 70%)"},
       "results": {"holdout_rejected": ho, "calib_rejected": ca, "n1_rejected": n1r,
                   "val_rejected": int(rej_val.sum()), "val_correct_after_rejection": correct_kept},
       "decision": "PASS" if ok else "FAIL"}
s = json.dumps(det)
with open(OUT_JSON, "w") as f: f.write(s)
print("detector saved:", OUT_JSON, "sha256", hashlib.sha256(s.encode()).hexdigest())
print("===== END =====")